# Get scraped scotus data

We scraped the scotus docket data for all dockets between 2001 and 2024, to the extent they are available on the scotus website.

This notebook outlines the steps undertook to parse relevant information from the scraped pages for linking the appellate chain (scotus to the lower court). The final output is scotus_data.csv

# Import libraries

In [1]:
import os

import numpy as np
import pandas as pd

from parse_utils import parse_scotus_file

# Use the helper functions to get the data & clean as needed

In [2]:
%%time

root_folder = "scotus_dockets/scraped/"
count = 0
jsons = []

for dirpath, dirnames, filenames in os.walk(root_folder):
    # Clean up subdirectories
    dirnames[:] = [d for d in dirnames if not d.endswith('.ipynb_checkpoints') and not d.endswith('.DS_Store')]

    # Skip files ending with .ipynb_checkpoints
    filenames = [f for f in filenames if not f.endswith('.ipynb_checkpoints') and not f.endswith('.DS_Store')]
    if not filenames:
        continue

    # Get current folder name
    folder_name = os.path.basename(dirpath)
    json_filename = f"data/scraped_scotus/{folder_name}.json"
    print("--- Processing folder:", folder_name)

    results = {}
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        results[filename] = parse_scotus_file(file_path)
        count += 1

    df = pd.DataFrame.from_dict(results, orient='index').reset_index().rename(columns={'index': 'filename'})
    df.to_json(json_filename)
    jsons.append(f"{folder_name}.json")
    print(f"Saved results to: {json_filename}")

print(f"\nProcessed {count} files and saved JSONs: {', '.join(sorted(jsons))}")

--- Processing folder: 03
Saved results to: data/scraped_scotus/03.json
--- Processing folder: 04
Saved results to: data/scraped_scotus/04.json
--- Processing folder: 05
Saved results to: data/scraped_scotus/05.json
--- Processing folder: 02
Saved results to: data/scraped_scotus/02.json
--- Processing folder: 20
Saved results to: data/scraped_scotus/20.json
--- Processing folder: 18
Saved results to: data/scraped_scotus/18.json
--- Processing folder: 11
Saved results to: data/scraped_scotus/11.json
--- Processing folder: 16
Saved results to: data/scraped_scotus/16.json
--- Processing folder: 17
Saved results to: data/scraped_scotus/17.json
--- Processing folder: 10
Saved results to: data/scraped_scotus/10.json
--- Processing folder: 19
Saved results to: data/scraped_scotus/19.json
--- Processing folder: 21
Saved results to: data/scraped_scotus/21.json
--- Processing folder: 07
Saved results to: data/scraped_scotus/07.json
--- Processing folder: 00
Saved results to: data/scraped_scotus/

# Combine the individual csvs for each year into one large csv

In [3]:
df_list = []

for json_file in jsons:
    df = pd.read_json(f"data/scraped_scotus/{json_file}")
    df_list.append(df)

# Combine all DataFrames
scotus_df = pd.concat(df_list, ignore_index=True)

# Rename the case_number column
scotus_df = scotus_df.rename(columns={'case_number': 'docket_number'})

# Get the year and case id for the particular year
scotus_df["year"] = scotus_df["docket_number"].str.split("-").str[0]
scotus_df["case_num"] = scotus_df["docket_number"].str.split("-").str[1].astype(int)

len(scotus_df)

165583

In [4]:
scotus_df.head()

,filename,docket_number,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
0,03-6084.htm,03-6084,2003-08-27,"Ronald Lee Smith, Petitioner v. Larry Reid, Wa...",United States Court of Appeals for the Tenth C...,(03-1016),03-1016,2003-05-16,2003-07-09,03,6084
1,03-10616.htm,03-10616,2004-05-28,"Jimmy Walker, Petitioner v. Florida","District Court of Appeal of Florida, Fourth Di...",(4D02-4272),4D02-4272,2004-04-21,None,03,10616
2,03-777.htm,03-777,2003-11-28,"Willie R. Flint, Petitioner v. ABB Inc., fka A...",United States Court of Appeals for the Elevent...,(02-15029),02-15029,2003-07-21,2003-08-27,03,777
3,03-10170.htm,03-10170,2004-05-05,"Loyda Lugones, Petitioner v. United States",United States Court of Appeals for the Elevent...,(02-12984),02-12984,2004-01-26,None,03,10170
4,03-8917.htm,03-8917,2004-02-19,"Ray Charles Smith, Petitioner v. United States",United States Court of Appeals for the Ninth C...,(03-10003),03-10003,2003-11-13,None,03,8917


# Confirm case numbers are bound between 1-3000 and between 5000-15000 for all years

In [5]:
# Define valid bounds
lower_range = (1, 3000)
upper_range = (5000, 15000)

# Function to check if all case numbers fall within the allowed ranges
def is_year_valid(series):
    return series.dropna().apply(
        lambda x: lower_range[0] <= x <= lower_range[1] or upper_range[0] <= x <= upper_range[1]
    ).all()

# Group by year and check validity
validity_by_year = scotus_df.groupby("year")["case_num"].apply(is_year_valid)
invalid_years = validity_by_year[~validity_by_year]

assert invalid_years.empty

# Check for years where the max for each range might have been cut off

In [6]:
lower_range = (1, 3000)
upper_range = (5000, 15000)

grouped = scotus_df.groupby("year")

lower_max = grouped.apply(lambda g: g.loc[g["case_num"].between(*lower_range), "case_num"].max())

lower_max

/var/folders/d9/3h7m7wc52kv6fgxmbyd8s0940000gn/T/ipykernel_88889/1851348596.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lower_max = grouped.apply(lambda g: g.loc[g["case_num"].between(*lower_range), "case_num"].max())


year
00    1954
01    1886
02    1869
03    1722
04    1741
05    1671
06    1723
07    1614
08    1596
09    1580
10    1558
11    1552
12    2042
13    1976
14    1544
15    1546
16    1550
17    1718
18    1593
19    1478
20    1829
21    1611
22    1252
23    1375
24    1888
99     927
dtype: int64

In [7]:
upper_max = grouped.apply(lambda g: g.loc[g["case_num"].between(*upper_range), "case_num"].max())

upper_max

/var/folders/d9/3h7m7wc52kv6fgxmbyd8s0940000gn/T/ipykernel_88889/1495430752.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  upper_max = grouped.apply(lambda g: g.loc[g["case_num"].between(*upper_range), "case_num"].max())


year
00    10897.0
01    11037.0
02    11386.0
03    11092.0
04    10755.0
05    11846.0
06    12132.0
07    11627.0
08    11142.0
09    11576.0
10    12000.0
11    11160.0
12    11005.0
13    11057.0
14    10488.0
15     9926.0
16     9755.0
17     9595.0
18     9847.0
19     8930.0
20     8477.0
21     8288.0
22     7907.0
23     7848.0
24     7529.0
99        NaN
dtype: float64

# Confirm no missing docket numbers

In [8]:
assert len(scotus_df[scotus_df["docket_number"].isna()]) == 0

# Confirm no duplicate docket numbers

In [9]:
assert len(scotus_df[scotus_df.duplicated(subset=['docket_number'])]) == 0

# Confirm no missing case titles and docket dates

In [10]:
assert len(scotus_df[scotus_df["case_title"].isnull()]) == 0

In [11]:
assert len(scotus_df[scotus_df["docket_date"].isnull()]) == 0

# Check for number of missing lower court name or lower court case numbers for each year

Manually verify a sample of those with lower court but without lower court case numbers is due to missing lower court case number info from scotus

In [12]:
scotus_df[(~scotus_df["lower_court"].isnull()) & (scotus_df["lower_court_case_numbers_raw"].isnull())]

,filename,docket_number,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
1906,03-6959.htm,03-6959,2003-10-20,"Kennith Charles Huff, Petitioner v. Virginia",Supreme Court of Virginia,None,None,2002-11-06,2003-01-10,03,6959
4193,03-5900.htm,03-5900,2003-08-14,"John M. Miller, Petitioner v. South Carolina",Supreme Court of South Carolina,None,None,2003-04-29,None,03,5900
12906,04-5977.htm,04-5977,2004-08-24,"Yaqub Hameed Muwakkil, Petitioner v. Virginia,...",Supreme Court of Virginia,None,None,2004-05-28,None,04,5977
15647,05-10610.htm,05-10610,2006-04-26,"Billy Williams, Petitioner v. Virginia",Supreme Court of Virginia,None,None,2006-02-23,None,05,10610
24332,02-10272.htm,02-10272,2003-04-22,"Larry D. Marvel, Petitioner v. Delaware",Supreme Court of Delaware,None,None,None,None,02,10272
...,...,...,...,...,...,...,...,...,...,...,...
147816,13-9507.htm,13-9507,2014-04-02,"Cynthia E. Collie, Petitioner v. South Carolin...",Supreme Court of South Carolina,None,None,2012-05-02,2013-06-19,13,9507
148761,13-559.htm,13-559,2013-11-05,"Heather Lukashin, et vir, Petitioners v. Allia...",Supreme Court of Washington,None,None,2013-06-18,None,13,559
149522,13-10380.htm,13-10380,2014-06-03,"Cynthia E. Collie, Petitioner v. South Carolin...",Supreme Court of South Carolina,None,None,2013-10-17,2014-01-09,13,10380
150033,13-9818.htm,13-9818,2014-04-22,"Michael Charles Ward, Petitioner v. Michigan P...",Supreme Court of Michigan,None,None,2014-03-13,None,13,9818


Confirm the only instance where we have the lower court case number but did not have the lower court was due to missing info from scotus

In [13]:
scotus_df[(scotus_df["lower_court"].isnull()) & (~scotus_df["lower_court_case_numbers_raw"].isnull())]

,filename,docket_number,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
111577,01-7018.htm,01-7018,2001-10-25,"In Re Mitchell T. Mincey, Petitioner v.",None,(01-16017),01-16017,None,None,01,7018


Confirm majority of the cases with missing lower court information are cases "In Re"

In [14]:
subset = scotus_df[scotus_df["lower_court_case_numbers_raw"].isnull()]
starts_with_in_re = subset["case_title"].str.startswith("In Re")

num_in_re = starts_with_in_re.sum()
num_not_in_re = (~starts_with_in_re).sum()

print(f"Num Cases that start with 'In Re': {num_in_re} / {len(subset)}")
print(f"Num Cases that do NOT start with 'In Re': {num_not_in_re} / {len(subset)}")

Num Cases that start with 'In Re': 5146 / 5217
Num Cases that do NOT start with 'In Re': 71 / 5217


Manually verify a sample of those without lower court and not In Re cases is due to missing lower court info from scotus

In [15]:
subset[~starts_with_in_re]

,filename,docket_number,docket_date,case_title,lower_court,lower_court_case_numbers_raw,lower_court_case_numbers,lower_court_decision_date,lower_court_rehearing_denied_date,year,case_num
1206,03-5429.htm,03-5429,2003-07-22,"Richard W. Cooey, II, Petitioner v. Ohio",None,None,None,None,None,03,5429
1906,03-6959.htm,03-6959,2003-10-20,"Kennith Charles Huff, Petitioner v. Virginia",Supreme Court of Virginia,None,None,2002-11-06,2003-01-10,03,6959
3338,03-6747.htm,03-6747,2003-10-06,"M. K. B., Petitioner v. Warden, et al.",None,None,None,None,None,03,6747
4193,03-5900.htm,03-5900,2003-08-14,"John M. Miller, Petitioner v. South Carolina",Supreme Court of South Carolina,None,None,2003-04-29,None,03,5900
12906,04-5977.htm,04-5977,2004-08-24,"Yaqub Hameed Muwakkil, Petitioner v. Virginia,...",Supreme Court of Virginia,None,None,2004-05-28,None,04,5977
...,...,...,...,...,...,...,...,...,...,...,...
147816,13-9507.htm,13-9507,2014-04-02,"Cynthia E. Collie, Petitioner v. South Carolin...",Supreme Court of South Carolina,None,None,2012-05-02,2013-06-19,13,9507
148761,13-559.htm,13-559,2013-11-05,"Heather Lukashin, et vir, Petitioners v. Allia...",Supreme Court of Washington,None,None,2013-06-18,None,13,559
149522,13-10380.htm,13-10380,2014-06-03,"Cynthia E. Collie, Petitioner v. South Carolin...",Supreme Court of South Carolina,None,None,2013-10-17,2014-01-09,13,10380
150033,13-9818.htm,13-9818,2014-04-22,"Michael Charles Ward, Petitioner v. Michigan P...",Supreme Court of Michigan,None,None,2014-03-13,None,13,9818


# Save the data for future use

In [16]:
scotus_df.to_json("data/scotus_data.json")

In [17]:
len(scotus_df)

165583